# Dataset Splitting Pipeline

## Overview

This notebook performs dataset splitting for a multimodal breast cancer Recurrence-Free Survival (RFS) prediction dataset.

The objective is to divide the constructed multimodal dataset into independent training, validation, and testing subsets while preserving the distribution of RFS recurrence events across all splits.

A fixed random seed is used to ensure reproducibility.

---

## Dataset Splitting Strategy

The complete multimodal dataset is divided into three subsets:

- Training set: 70%
- Validation set: 15%
- Testing set: 15%

The splitting process uses stratified sampling based on the RFS event variable to maintain similar recurrence-event distributions across all subsets.

Random seed:

- 42

---

## Pipeline

1. Load the final multimodal RFS dataset.

2. Initialize random seed for reproducibility.

3. Perform stratified train-validation-test splitting:

   - Training dataset (70%)
   - Validation dataset (15%)
   - Testing dataset (15%)

4. Verify dataset distributions:

   - Number of patients
   - Number of recurrence events
   - Event ratio
   - Patient distribution across subsets

5. Export split datasets:

   - train_rfs.csv
   - validation_rfs.csv
   - test_rfs.csv

6. Save patient ID lists for each split:

   - train_rfs_patient_ids.csv
   - validation_rfs_patient_ids.csv
   - test_rfs_patient_ids.csv

---

## Dataset Split Results

The final multimodal dataset contains:

Total patients:

- 357 patients


Dataset distribution:

| Split | Patients | Percentage |
|------|----------|------------|
| Training | 249 | 70% |
| Validation | 54 | 15% |
| Testing | 54 | 15% |


RFS outcome distribution:

- Total recurrence events: 91
- Censored cases: 266

---

## Quality Control and Validation

The generated splits were verified to ensure:

- No duplicated patients
- No patient overlap between subsets
- Consistent dataset structure
- Preserved RFS event distribution
- Reproducible splitting using random seed 42

---

## Output

The generated datasets are stored in:

dataset_split_rfs/


The resulting datasets are ready for:

- Genomic feature selection
- Multimodal survival model development
- Transformer-based RFS prediction
- Classical survival model comparison

##Imports

In [ ]:
# ============================================================
# IMPORTS
# ============================================================

import os
import random

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

print("Libraries loaded successfully.")

Libraries loaded successfully.


##Mount Google Drive

In [ ]:
# ============================================================
# MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


##Project Paths

In [ ]:
# ============================================================
# PROJECT PATHS
# ============================================================

BASE_DIR = "/content/drive/MyDrive/TCGA_BRCA"

MASTER_DATASET = os.path.join(
    BASE_DIR,
    "multimodal_rfs_dataset",
    "multimodal_master_RFS_dataset.csv"
)

SAVE_DIR = os.path.join(
    BASE_DIR,
    "dataset_split_rfs"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

print("="*70)
print("PROJECT PATHS")
print("="*70)

print("Master dataset:")
print(MASTER_DATASET)

print("\nOutput folder:")
print(SAVE_DIR)

PROJECT PATHS
Master dataset:
/content/drive/MyDrive/TCGA_BRCA/multimodal_rfs_dataset/multimodal_master_RFS_dataset.csv

Output folder:
/content/drive/MyDrive/TCGA_BRCA/dataset_split_rfs


##Load Master Dataset

In [ ]:
# ============================================================
# LOAD MASTER DATASET
# ============================================================

multimodal = pd.read_csv(
    MASTER_DATASET
)

print("="*70)
print("MASTER DATASET")
print("="*70)

print("Shape:")
print(multimodal.shape)

print("\nColumns:")
print(len(multimodal.columns))

display(
    multimodal.head()
)

MASTER DATASET
Shape:
(357, 1024)

Columns:
1024


,patient_id,file_uuid,years_to_birth,Tumor_purity,pathologic_stage,pathology_T_stage,pathology_N_stage,pathology_M_stage,number_of_lymph_nodes,radiation_therapy,...,SLC1A6,PTGDS,TFCP2L1,ZIC5,CRYAB,EMILIN3,TNNT2,INA,event,survival_time
0,TCGA-AC-A3BB,00ab866f-a0a3-44c0-ad14-c9dc833239a7,-0.962155,-1.473020,3,3,2,0,0.663363,1,...,-0.365232,1.614857,-0.969645,-0.384022,0.737494,0.452037,-0.530162,0.002997,0,987.0
1,TCGA-A8-A06O,01c79d06-a4a7-47a7-bb8b-b98ce6652c99,0.106767,0.065894,1,1,0,0,-0.269000,0,...,-0.580847,-0.200473,0.987724,1.405035,0.375263,-1.019845,-0.681348,-0.674596,0,396.0
2,TCGA-B6-A0WZ,02732da3-5144-4680-b6ee-cb8563125493,-0.656749,0.958908,2,2,1,0,2.061906,1,...,-0.580847,-0.846220,-0.451058,-0.384022,-0.795755,-0.456894,-1.043650,-0.253540,0,6292.0
3,TCGA-E2-A1L9,0279f555-9738-41e0-b89f-ded25829adc0,-1.420264,0.400312,2,1,1,0,-0.269000,1,...,-0.580847,-0.905408,-1.886017,-0.186071,0.276367,-0.705331,-1.043650,0.034138,0,598.0
4,TCGA-AC-A5XU,032fc1ed-c515-4b26-a229-f67c1eb6fdc3,1.175689,0.375897,2,3,0,0,-0.502090,1,...,-0.351638,-0.997118,-1.802918,-0.599734,-0.579521,-1.000917,-1.043650,-0.338875,0,455.0


In [ ]:
print(multimodal.columns.tolist())

['patient_id', 'file_uuid', 'years_to_birth', 'Tumor_purity', 'pathologic_stage', 'pathology_T_stage', 'pathology_N_stage', 'pathology_M_stage', 'number_of_lymph_nodes', 'radiation_therapy', 'histological_type_infiltratinglobularcarcinoma', 'histological_type_medullarycarcinoma', 'histological_type_metaplasticcarcinoma', 'histological_type_mixedhistology(pleasespecify)', 'histological_type_mucinouscarcinoma', 'histological_type_other,specify', 'PAM50_Her2', 'PAM50_LumA', 'PAM50_LumB', 'race_blackorafricanamerican', 'race_white', 'ethnicity_nothispanicorlatino', 'CLEC3A', 'CPB1', 'SCGB2A2', 'SCGB1D2', 'TFF1', 'GSTM1', 'PIP', 'S100A7', 'MUCL1', 'CYP2B7P1', 'ANKRD30A', 'PRAME', 'CYP4Z1', 'KCNJ3', 'AGR3', 'HMGCS2', 'SERPINA6', 'TFAP2B', 'MUC6', 'DHRS2', 'SLC30A8', 'UGT2B11', 'VSTM2A', 'COL2A1', 'C4orf7', 'TAT', 'ADIPOQ', 'ADH1B', 'CALML5', 'GP2', 'MYBPC1', 'GABRP', 'KRT14', 'CEACAM5', 'MUC5B', 'TFF3', 'C1orf64', 'SOX10', 'GRIA2', 'KRT5', 'CRABP1', 'GSTT1', 'SYT13', 'STAC2', 'CST9', 'LTF', 

##Dataset Quality Control

In [ ]:
# ============================================================
# DATASET QUALITY CONTROL
# ============================================================

print("="*70)
print("QUALITY CONTROL")
print("="*70)


print("Patients:")
print(
    multimodal["patient_id"].nunique()
)


print("\nRows:")
print(
    len(multimodal)
)


print("\nDuplicate patients:")
print(
    multimodal["patient_id"].duplicated().sum()
)


print("\nMissing RFS event:")
print(
    multimodal["event"].isna().sum()
)


print("\nMissing RFS time:")
print(
    multimodal["survival_time"].isna().sum()
)


print("\nMissing pathology UUID:")
print(
    multimodal["file_uuid"].isna().sum()
)


print("\nRFS event distribution:")
print(
    multimodal["event"].value_counts()
)

QUALITY CONTROL
Patients:
357

Rows:
357

Duplicate patients:
0

Missing RFS event:
0

Missing RFS time:
0

Missing pathology UUID:
0

RFS event distribution:
event
0    266
1     91
Name: count, dtype: int64


##Fix Random Seed

In [ ]:
# ============================================================
# RANDOM SEED
# ============================================================

RANDOM_SEED = 42

random.seed(RANDOM_SEED)

np.random.seed(RANDOM_SEED)

print("="*70)
print("RANDOM SEED")
print("="*70)

print(RANDOM_SEED)

RANDOM SEED
42


##Stratified Dataset Split

In [ ]:
# ============================================================
# STRATIFIED TRAIN / VALIDATION / TEST SPLIT
# ============================================================

print("="*70)
print("CREATING DATASET SPLITS")
print("="*70)


train_df, temp_df = train_test_split(

    multimodal,

    test_size=0.30,

    stratify=multimodal["event"],

    random_state=RANDOM_SEED
)



validation_df, test_df = train_test_split(

    temp_df,

    test_size=0.50,

    stratify=temp_df["event"],

    random_state=RANDOM_SEED
)



print("Train:")
print(train_df.shape)

print()

print("Validation:")
print(validation_df.shape)

print()

print("Test:")
print(test_df.shape)

CREATING DATASET SPLITS
Train:
(249, 1024)

Validation:
(54, 1024)

Test:
(54, 1024)


##Verify Split

In [ ]:
# ============================================================
# VERIFY DATASET SPLITS
# ============================================================

print("=" * 70)
print("VERIFYING DATASET SPLITS")
print("=" * 70)


def summarize_split(df, name):

    total = len(df)

    events = int(df["event"].sum())

    non_events = total - events

    ratio = events / total

    print(f"\n{name}")

    print("-" * 40)

    print(f"Patients      : {total}")

    print(f"Events        : {events}")

    print(f"Non-events    : {non_events}")

    print(f"Event Ratio   : {ratio:.4f}")


summarize_split(train_df, "TRAIN")

summarize_split(validation_df, "VALIDATION")

summarize_split(test_df, "TEST")

VERIFYING DATASET SPLITS

TRAIN
----------------------------------------
Patients      : 249
Events        : 63
Non-events    : 186
Event Ratio   : 0.2530

VALIDATION
----------------------------------------
Patients      : 54
Events        : 14
Non-events    : 40
Event Ratio   : 0.2593

TEST
----------------------------------------
Patients      : 54
Events        : 14
Non-events    : 40
Event Ratio   : 0.2593


##Save Train Dataset

In [ ]:
# ============================================================
# SAVE TRAIN DATASET
# ============================================================

train_path = os.path.join(
    SAVE_DIR,
    "train_rfs.csv"
)

train_df.to_csv(
    train_path,
    index=False
)

print("=" * 70)
print("TRAIN DATASET SAVED")
print("=" * 70)

print(train_path)

TRAIN DATASET SAVED
/content/drive/MyDrive/TCGA_BRCA/dataset_split_rfs/train_rfs.csv


##Save Validation Dataset

In [ ]:
# ============================================================
# SAVE VALIDATION DATASET
# ============================================================

validation_path = os.path.join(
    SAVE_DIR,
    "validation_rfs.csv"
)

validation_df.to_csv(
    validation_path,
    index=False
)

print("=" * 70)
print("VALIDATION DATASET SAVED")
print("=" * 70)

print(validation_path)

VALIDATION DATASET SAVED
/content/drive/MyDrive/TCGA_BRCA/dataset_split_rfs/validation_rfs.csv


##Save Test Dataset

In [ ]:
# ============================================================
# SAVE TEST DATASET
# ============================================================

test_path = os.path.join(
    SAVE_DIR,
    "test_rfs.csv"
)

test_df.to_csv(
    test_path,
    index=False
)

print("=" * 70)
print("TEST DATASET SAVED")
print("=" * 70)

print(test_path)

TEST DATASET SAVED
/content/drive/MyDrive/TCGA_BRCA/dataset_split_rfs/test_rfs.csv


##Save Patient ID Lists

In [ ]:
# ============================================================
# SAVE PATIENT ID LISTS
# ============================================================

train_df[["patient_id"]].to_csv(
    os.path.join(SAVE_DIR, "train_rfs_patient_ids.csv"),
    index=False
)

validation_df[["patient_id"]].to_csv(
    os.path.join(SAVE_DIR, "validation_rfs_patient_ids.csv"),
    index=False
)

test_df[["patient_id"]].to_csv(
    os.path.join(SAVE_DIR, "test_rfs_patient_ids.csv"),
    index=False
)

print("=" * 70)
print("PATIENT ID LISTS SAVED")
print("=" * 70)

print(os.listdir(SAVE_DIR))

PATIENT ID LISTS SAVED
['train_rfs.csv', 'validation_rfs.csv', 'test_rfs.csv', 'train_rfs_patient_ids.csv', 'validation_rfs_patient_ids.csv', 'test_rfs_patient_ids.csv']


##Save Split Statistics

In [ ]:
# ============================================================
# SAVE SPLIT STATISTICS
# ============================================================

statistics = pd.DataFrame({
    "Split": [
        "Train",
        "Validation",
        "Test"
    ],
    "Patients": [
        len(train_df),
        len(validation_df),
        len(test_df)
    ],
    "Events": [
        int(train_df["event"].sum()),
        int(validation_df["event"].sum()),
        int(test_df["event"].sum())
    ],
    "Non_events": [
        len(train_df) - int(train_df["event"].sum()),
        len(validation_df) - int(validation_df["event"].sum()),
        len(test_df) - int(test_df["event"].sum())
    ],
    "RFS_event_ratio": [
        train_df["event"].mean(),
        validation_df["event"].mean(),
        test_df["event"].mean()
    ]
})

statistics.to_csv(
    os.path.join(
        SAVE_DIR,
        "rfs_split_statistics.csv"
    ),
    index=False
)

print("=" * 70)
print("SPLIT STATISTICS")
print("=" * 70)

display(statistics)

SPLIT STATISTICS


,Split,Patients,Events,Non_events,RFS_event_ratio
0,Train,249,63,186,0.253012
1,Validation,54,14,40,0.259259
2,Test,54,14,40,0.259259


##Final Validation

In [ ]:
# ============================================================
# FINAL VALIDATION
# ============================================================

print("=" * 70)
print("FINAL VALIDATION")
print("=" * 70)

all_patients = (
    set(train_df["patient_id"])
    |
    set(validation_df["patient_id"])
    |
    set(test_df["patient_id"])
)

train_val_overlap = (
    set(train_df["patient_id"])
    &
    set(validation_df["patient_id"])
)

train_test_overlap = (
    set(train_df["patient_id"])
    &
    set(test_df["patient_id"])
)

validation_test_overlap = (
    set(validation_df["patient_id"])
    &
    set(test_df["patient_id"])
)

print("Unique patients:", len(all_patients))

print("\nTrain duplicates:",
      train_df["patient_id"].duplicated().sum())

print("Validation duplicates:",
      validation_df["patient_id"].duplicated().sum())

print("Test duplicates:",
      test_df["patient_id"].duplicated().sum())

print("\nTrain ↔ Validation overlap:",
      len(train_val_overlap))

print("Train ↔ Test overlap:",
      len(train_test_overlap))

print("Validation ↔ Test overlap:",
      len(validation_test_overlap))

print("\nMissing events:")

print(
    train_df["event"].isna().sum() +
    validation_df["event"].isna().sum() +
    test_df["event"].isna().sum()
)

print("\nMissing survival:")

print(
    train_df["survival_time"].isna().sum() +
    validation_df["survival_time"].isna().sum() +
    test_df["survival_time"].isna().sum()
)


FINAL VALIDATION
Unique patients: 357

Train duplicates: 0
Validation duplicates: 0
Test duplicates: 0

Train ↔ Validation overlap: 0
Train ↔ Test overlap: 0
Validation ↔ Test overlap: 0

Missing events:
0

Missing survival:
0


##Notebook Summary

In [ ]:
# ============================================================
# DATASET SPLIT SUMMARY
# ============================================================

print("=" * 70)
print("DATASET SPLITTING COMPLETED")
print("=" * 70)

print(f"Master dataset        : {len(multimodal)} patients")

print(f"Training set          : {len(train_df)}")

print(f"Validation set        : {len(validation_df)}")

print(f"Testing set           : {len(test_df)}")

print(f"RFS recurrence events : {multimodal['event'].sum()}")

print(f"Random seed           : {RANDOM_SEED}")

print(f"Split strategy        : Stratified")

print(f"Train ratio           : 70%")

print(f"Validation ratio      : 15%")

print(f"Test ratio            : 15%")

print("\nOutput directory:")

print(SAVE_DIR)

print("\nNotebook completed successfully.")

DATASET SPLITTING COMPLETED
Master dataset        : 357 patients
Training set          : 249
Validation set        : 54
Testing set           : 54
RFS recurrence events : 91
Random seed           : 42
Split strategy        : Stratified
Train ratio           : 70%
Validation ratio      : 15%
Test ratio            : 15%

Output directory:
/content/drive/MyDrive/TCGA_BRCA/dataset_split_rfs

Notebook completed successfully.
